
## Subset interpretation
`Type ~ (A, Asub)` not that we interpret "bare" type as anything.

So valid families are functions/dicts to sets such that when the key is in it's valid subset, the output is well formed Asub <= A


Nat vs Int is too big

Fin(n,N) as subsets?

True = {x : Bool | x = true}
UnitVoid = {x : Unit | false}

What would be ill formed?

Maybe the map is not require to be defined (can be partial) when arguments are out of required subsets


ok, I don't really need the iterated construction.

Rather than materializing `good`, could have `valid : Callable[object, bool]` predicate.


In [ ]:
from dataclasses import dataclass, field
type Env = tuple

@dataclass
class SubSet[T]:
    total : set[T]
    good : set[T]
    def __init__(self, total, good=None):
        if good is None:
            good = total
        else:
            assert good <= total
        self.total = total
        self.good = good

type Ctx = SubSet[Env]

@dataclass
class Family:
    good : set[Env]
    data : dict[Env, SubSet[object]]

    def ext_eq(self, other):
        return self.good == other.good and all(self.data[env] == other.data[env] for env in self.good)

# intensional equality is f1.data == f2.data (and f1.good == f2.good). Extensional is forall 





In [ ]:

@dataclass
class Ctx:
    good: set[Env]
    total: set[Env]
    def __post_init__(self):
        assert self.good <= self.total


In [ ]:
from dataclasses import dataclass,field

type Family = tuple[list[set[object]], dict[tuple[object], tuple[set, set]]]
# no a factored construction? It _has_ to be a trie. And we have an outer set and

type Family = tuple[set, dict[object, Family | tuple[set, set]]]
# or at every stage have a bool saying if in or out of the subset

type Family = dict[object, tuple[bool, Family | tuple[set, set]]]

@dataclass
class Family:
    argsubs : list[set]
    f : dict[object, tuple[set, set]]

    def subtype(self, P : Section):
        assert self.argsubs == P.argsubs
        return Family(self.argsubs, )

Bool = Family([], {() : ({True, False}, {True, False})})




Ok. But actually I wanted to write a post using taint

Taint exposes some of the computational character of _how_ you got the result.

https://en.wikipedia.org/wiki/Taint_checking

Taint without values at all to start off? A pure theory of taint. taint is a static analysis so we don't need to expand into all environements

```python
n |- t
```

Taint also feels like some kind of free construction, like dolan was saying? Free semilattices are basically sets of the generators.
free commutative semigroup/monoid is multisets. Adding idempotency basically makes it a set.


When you thin a section, it will retain that thinning

When you thin a type, it needs to enumerate all the thinnings that could work

`if x then 3 else 3` is tainted by x despite not really depending on it in an extensional way.

`x * 0 = 0` extensionally, but not in the taint model.

`x * 0` is a stuck term in typical left match definition. `0 * x` is not. But still tainted? Maybe. These are different definitions of *

`x & false` is a nicer example that is finite and exhibits the same issues. One possible rule of the game is that the definition is not allowed to look at taintedness of subexpressions to make choices.

Taint ~ Thinning ~ free variable analysis.
We might consider some variables tainted and others not, but the "free" version of this tracks which variables go into which stuff.

I guess it was a bit goofy to need groupby. Trie form will really be just 

Getting into the value of the trie sucks.
Why can't I just keep it in flat form.

Kind of the internalizing moves to Pi can be just one step of the ctx -> Pi



In [ ]:
from dataclasses import dataclass,field
from frozendict import frozendict
from typing import Union
# Trie form
#type Trie = tuple[dict[object, Trie], int] # int is how deep to go

# Or use special class as sigil
@dataclass(frozen=True)
class Trie:
    data : frozendict[object, Union[object,"Trie"]]

    def flatten(self) -> dict[tuple[object, ...], int]:
        return { (k,) + suffix : v for k, v in self.data.items() for suffix, v in (v.flatten().items() if isinstance(v, Trie) else {() : v}.items()) }
    
    @staticmethod
    def from_flat(flat : dict[tuple[object, ...]]) -> "Trie":
        root = {}
        for k, v in flat.items():
            d = root
            for part in k:
                if (part,) not in d:
                    d[(part,)] = {}
                d = d[(part,)]
            d[()] = v
        return Trie(frozendict(root))

Trie.from_flat({(1,2,3) : 42, (1,2,4) : 43})






Trie(data=frozendict.frozendict({(1,): {(2,): {(3,): {(): 42}, (4,): {(): 43}}}}))

In [ ]:
#type TaintFun = tuple[int, frozenset[int]]
type TaintFun = list[bool]  # which ones actually show up. This is exactly a thinning... Taint kind of tracks minimal context needed. Huh.

type Family =  tuple[InCtx[object], set[object], Thin] # What contexts does the type make sense in?
type Section = tuple[InCtx[object], Thin] # InCtx[tuple[object, Thin]] ? Thin is kind of static so shouldn't be env dependent.
def idthin(n):
    return [True] * n

Bool = ({(): {True, False}}, idthin(0))

# In the set semantics
ArrUnitUnit0 : Family = {(): {frozendict({() : ()})}}
idUnit : Section = {() : {() : ()}}

idUnitTaint_tainted : Section = ({() : ({() : ()}, True)}, idthin(0))
idUnitTaint_untainted : Section = ({() : ({() : ()}, False)}, idthin(0))
# Arrow values come packaged with whether they are tainted by the argument or not
# But then we have a more recursive looking thing. Where taint can depend on previous values
# Maybe the thin needs to be composed. So False, means we thin from the outerthinnning
# But True means we may still thin or not in the inner stuff.
# So maybe I really do need InCtx to be a trie to better match Arrow?
# So maybe my boolean should be a full Thin


In [ ]:
type Ctx = int
type InCtx[T] = tuple[int, T]

type Family = InCtx[set[int]] # free taint
type Section = InCtx[set[int]] # ?  which things tainted us?
def weak(t : Thin, x : InCtx)